# TCCT S86F - Revealed S84/S85 Regression

Run the exact revealed S84 and S85 counterfactual test generators with the frozen S86E K=33 candidate. This is regression, not a new blind test. No selection, policy edit, retuning, or S87 data is allowed.


In [ ]:
ClearAll["Global`*"];
$HistoryLength=0;

expectedFrozenModelHash79A=
"d6477c370436d09cf3e8cfc8530decd13ebf8bb79120362146ecb419f9d6a6c4";
expectedTopologySpecHash79A=
"b108aa7f2c59c395056fb9124e7d5d9074e39e5e124d0d98011800433e0dd95b";
expectedTopologyImplementationHash79A=
"f27d1ce683317d35a5e9da01536d95c706065810c27462e336cdb76f4a2dab52";
expectedS79BlindResultHash79A=
"b1ebf089e43121a61d4181139d4ba0a8970dc646a7a41425f861cff2e1f8661d";

frozen75D=<|
"Stage"->"S75D",
"Name"->"ValidatedPolicyCompletion",
"Representation"->
"PairedRadius2Radius3WithParentChildCardinality",
"Params"->{-1,0,-1,-1,-1,0,1,-1},
"K"->5,
"TrainingPolicy"->{{3,1},{3,3},{4,3}},
"Policy"->{
{1,3},{2,2},{3,1},{3,2},{3,3},{4,3}
},
"AddedContinueCodes"->{{1,3},{2,2},{3,2}},
"PolicyCompletionUsesValidationLabels"->True,
"FrozenBeforeS76"->True
|>;

P59[a_,b_,n_Integer,s_Integer]:=If[
n<=0,
{DirectedEdge[a,b]},
DirectedEdge@@@Partition[
Join[{a},Range[s,s+n-1],{b}],
2,
1
]
];

A59[t_,fan_Integer,s_Integer]:=Module[{p1,p2,ls},
p1=s+1;
p2=s+2;
ls=Range[s+3,s+2+fan];
Join[
{
DirectedEdge[p2,p1],
DirectedEdge[p1,t]
},
Table[
DirectedEdge[ls[[i]],p2],
{i,Length[ls]}
]
]
];

T59[d_Integer,r_String,a_Integer,seed_Integer]:=Module[
{
bb,K,c,v,q,e,f={},ib,m,safe,u,dum,r1,r2,wrong,
main,perm,anc,i
},
bb=1000000000 seed;
K=bb+1;
c=Table[bb+100+i,{i,4}];
v=Table[bb+200+i,{i,4}];
q=Table[bb+300+i,{i,4}];
e=Flatten[
Table[
{
DirectedEdge[K,c[[i]]],
DirectedEdge[c[[i]],v[[i]]]
},
{i,4}
],
1
];
Do[
ib=bb+20000000 i;
m=ib+1;
safe=ib+2;
u=ib+3;
dum=ib+4;
r1=ib+10;
r2=ib+20;
wrong=c[[1+Mod[i,4]]];
main=Join[
P59[q[[i]],r1,d,ib+1000000],
P59[q[[i]],r2,d,ib+2000000],
{
DirectedEdge[r1,m],
DirectedEdge[r2,m]
},
P59[q[[i]],safe,d+1,ib+3000000]
];
perm=If[
r==="Continue",
{
DirectedEdge[m,c[[i]]],
DirectedEdge[safe,dum],
DirectedEdge[u,wrong]
},
{
DirectedEdge[m,wrong],
DirectedEdge[safe,c[[i]]],
DirectedEdge[u,dum]
}
];
anc=Join[
A59[m,i,bb+970000000+10000 i],
A59[c[[i]],i,bb+980000000+10000 i]
];
e=Join[e,main,perm,anc];
AppendTo[f,m],
{i,4}
];
{{Union[e],q,K,v,c,f},a}
];

Case59[d_Integer,a_Integer,r_String]:=
T59[d,r,a,59000000+100 d+a];

rw60={-9,6,-2,-2,4,2,-2,6,0};

Pack60[c_List]:=Module[
{x=c[[1]],a=c[[2]],e,v,pos,gi,go,pa,ch,id,od,
pin,pout,cin,cout},
e=x[[1]];
v=Union@Join[
Flatten[List@@@e],
x[[2]],
{x[[3]]},
x[[4]],
x[[5]],
x[[6]]
];
pos=AssociationThread[v,Range[Length[v]]];
gi=GroupBy[
Cases[e,DirectedEdge[u_,w_]:>{w,u}],
First->Last
];
go=GroupBy[
Cases[e,DirectedEdge[u_,w_]:>{u,w}],
First->Last
];
pa=(Lookup[pos,Lookup[gi,#,{}]]&)/@v;
ch=(Lookup[pos,Lookup[go,#,{}]]&)/@v;
id=Length/@pa;
od=Length/@ch;
pin=(Total[id[[#]]]&)/@pa;
pout=(Total[od[[#]]]&)/@pa;
cin=(Total[id[[#]]]&)/@ch;
cout=(Total[od[[#]]]&)/@ch;
{
pa,id,od,
Lookup[pos,x[[2,a]]],
Lookup[pos,x[[5]]],
a,
Length[v],
pin,pout,cin,cout,v
}
];

SigLevels61[c_List,rmax_Integer]:=Module[
{p,pa,id,od,n,ch,cur,nxt,levs,j,k,u},
p=Pack60[c];
pa=p[[1]];
id=p[[2]];
od=p[[3]];
n=p[[7]];
ch=Table[{},n];
Do[
Do[
If[u>0,ch[[u]]=Append[ch[[u]],j]],
{u,pa[[j]]}
],
{j,n}
];
cur=AssociationThread[
Range[n],
MapThread[List,{id,od}]
];
levs={cur};
Do[
nxt=AssociationThread[
Range[n],
Table[
{
Lookup[cur,j],
Sort[Lookup[cur,pa[[j]]]],
Sort[Lookup[cur,ch[[j]]]]
},
{j,n}
]
];
AppendTo[levs,nxt];
cur=nxt,
{k,rmax}
];
levs
];

PropagationSafetyCap78[c_List]:=Module[{n},
n=Pack60[c][[7]];
Max[64,2 n+8]
];

RejectTrace78[c_List]:=Module[
{
p,pa,id,od,q,n,pin,pout,cin,cout,h,z,h2,nz,j,k,
aps,ap,pz,o,zv,rf,rd,dd,pg,allow,active=0,
rej={},rounds=0,terminatedNaturally=False,safetyCap
},
p=Pack60[c];
pa=p[[1]];
id=p[[2]];
od=p[[3]];
q=p[[4]];
n=p[[7]];
pin=p[[8]];
pout=p[[9]];
cin=p[[10]];
cout=p[[11]];
safetyCap=Max[64,2 n+8];
h=ConstantArray[0,n];
z=ConstantArray[0,n];
h[[q]]=1;
For[
k=1,
k<=safetyCap,
k++,
rounds=k;
h2=ConstantArray[0,n];
nz=ConstantArray[0,n];
active=0;
Do[
aps=Select[
pa[[j]],
Function[u,u>0&&h[[u]]==1]
];
ap=Length[aps];
If[
ap>0,
pz=Max[z[[aps]]];
o=Boole[ap>1];
zv={0,1,2,1,3,1,3,1}[[1+2 pz+o]];
nz[[j]]=zv;
rf={
1,
Boole[zv>0],
zv,
Sign[pin[[j]]-id[[j]]],
Sign[pout[[j]]-id[[j]]],
Sign[cin[[j]]-id[[j]]],
Sign[cout[[j]]-od[[j]]],
Sign[id[[j]]-od[[j]]],
Sign[pout[[j]]-cin[[j]]]
};
rd=rw60.rf;
dd=1-cin[[j]]+2 cout[[j]];
pg=od[[j]]+2 Boole[zv>0]-zv;
If[rd>0&&dd<=0,AppendTo[rej,{k,j}]];
allow=If[rd>0,dd>0,pg>0];
If[
TrueQ[allow],
h2[[j]]=1;
active++
]
],
{j,n}
];
h=h2;
z=nz;
If[
active==0,
terminatedNaturally=True;
Break[]
]
];
<|
"Rejects"->rej,
"RoundsExecuted"->rounds,
"TerminatedNaturally"->terminatedNaturally,
"SafetyCap"->safetyCap,
"HitSafetyCap"->And[
!TrueQ[terminatedNaturally],
SameQ[rounds,safetyCap]
],
"ActiveAfterLastRound"->active,
"PackedNodeCount"->n
|>
];

DecisionStatePairsFromRejects78[c_List,rej_List]:=Module[
{levels,nodes},
If[Length[rej]==0,Return[{}]];
levels=SigLevels61[c,3];
nodes=rej[[All,2]];
DeleteDuplicates[
Map[
Function[node,
{
Lookup[levels[[3]],node],
Lookup[levels[[4]],node]
}
],
nodes
]
]
];

EncodeRows75[data_List,p_List,k_Integer]:=Module[{rec},
rec[s_]:=rec[s]=If[
MatchQ[s,{_Integer,_Integer}],
1+Mod[s[[1]]+2 s[[2]],k],
Module[
{z,ps,cs,parentSum,childSum,parentSquare,childSquare},
z=rec[s[[1]]];
ps=rec/@s[[2]];
cs=rec/@s[[3]];
parentSum=Total[ps-1];
childSum=Total[cs-1];
parentSquare=Total[(ps-1)^2];
childSquare=Total[(cs-1)^2];
1+Mod[
p[[1]]+
p[[2]](z-1)+
p[[3]]parentSum+
p[[4]]childSum+
p[[5]]parentSquare+
p[[6]]childSquare+
p[[7]]Length[ps]+
p[[8]]Length[cs],
k
]
]
];
Map[
Function[row,
Join[
KeyTake[row,{"Grammar","Depth","Answer","Target"}],
<|
"Codes"->DeleteDuplicates[
Map[
Function[pair,{rec[pair[[1]]],rec[pair[[2]]]}],
row["StatePairs"]
]
]
|>
]
],
data
]
];

DiamondIn72[c_List]:=Module[
{x=c[[1]],a=c[[2]],e,f,mx,next,new,m,incs,rm,
add,s1,s2,g,i,j},
e=x[[1]];
f=x[[6]];
mx=Max@Flatten[List@@@e];
next=mx+1;
new=e;
Do[
m=f[[i]];
incs=Cases[new,DirectedEdge[u_,v_]/;v===m:>{u,v}];
rm=DirectedEdge@@@incs;
add={};
Do[
s1=next;
s2=next+1;
g=next+2;
next=next+3;
add=Join[
add,
{
DirectedEdge[incs[[j,1]],s1],
DirectedEdge[incs[[j,1]],s2],
DirectedEdge[s1,g],
DirectedEdge[s2,g],
DirectedEdge[g,m]
}
],
{j,Length[incs]}
];
new=Join[Complement[new,rm],add],
{i,Length[f]}
];
{{Union[new],x[[2]],x[[3]],x[[4]],x[[5]],x[[6]]},a}
];

DoubleDiamondIn79[c_List]:=Module[
{
x=c[[1]],a=c[[2]],e,f,mx,next,new,m,incs,removed,
added,parent,s1,s2,g1,t1,t2,g2,i,j
},
e=x[[1]];
f=x[[6]];
mx=Max@Flatten[List@@@e];
next=mx+1;
new=e;
Do[
m=f[[i]];
incs=Cases[new,DirectedEdge[u_,v_]/;v===m:>{u,v}];
removed=DirectedEdge@@@incs;
added={};
Do[
parent=incs[[j,1]];
s1=next;
s2=next+1;
g1=next+2;
t1=next+3;
t2=next+4;
g2=next+5;
next=next+6;
added=Join[
added,
{
DirectedEdge[parent,s1],
DirectedEdge[parent,s2],
DirectedEdge[s1,g1],
DirectedEdge[s2,g1],
DirectedEdge[g1,t1],
DirectedEdge[g1,t2],
DirectedEdge[t1,g2],
DirectedEdge[t2,g2],
DirectedEdge[g2,m]
}
],
{j,Length[incs]}
];
new=Join[Complement[new,removed],added],
{i,Length[f]}
];
{{Union[new],x[[2]],x[[3]],x[[4]],x[[5]],x[[6]]},a}
];

Case79[
depth_Integer,
answer_Integer,
target_String
]:=DoubleDiamondIn79[
Case59[depth,answer,target]
];

topologySpec79=<|
"Topology"->"DoubleDiamondIn",
"TransformationScope"->
"EveryIncomingEdgeOfEachDecisionNode",
"Motif"->"TwoSerialPrivateDiamonds",
"CrossParentSharing"->False,
"DiamondsPerOriginalIncomingEdge"->2,
"ParentToDecisionPathLength"->5,
"NewNodesPerOriginalIncomingEdge"->6,
"ReplacementEdgesPerOriginalIncomingEdge"->9,
"EdgeDeltaPerOriginalIncomingEdge"->8,
"ReachabilityPreserved"->True,
"OriginalDecisionNodesPreserved"->True,
"TopologyUsedInS59ThroughS78"->False
|>;

topologySpecHash79=Hash[
Normal[topologySpec79],
"SHA256",
"HexString"
];

topologyImplementationHash79=Hash[
{DownValues[DoubleDiamondIn79],DownValues[Case79]},
"SHA256",
"HexString"
];

modelHash79A=Hash[
Normal[frozen75D],
"SHA256",
"HexString"
];

minimalKernelHash79A=Hash[
{
DownValues[P59],
DownValues[A59],
DownValues[T59],
DownValues[Case59],
OwnValues[rw60],
DownValues[Pack60],
DownValues[SigLevels61],
DownValues[PropagationSafetyCap78],
DownValues[RejectTrace78],
DownValues[DecisionStatePairsFromRejects78],
DownValues[EncodeRows75],
DownValues[DiamondIn72],
DownValues[DoubleDiamondIn79],
DownValues[Case79]
},
"SHA256",
"HexString"
];


ClearAll[
FindPrivateDiamond79B,
CanonicalizePrivateDiamonds79B,
CanonicalCase79B,
CaseByTopology79B,
DecisionIncomingEdgeCount79B,
ExpectedContractions79B
];

FindPrivateDiamond79B[e_List,protected_List]:=Module[
{
vertices,parents,children,harvest,ins,outs,s1,s2,
p1,p2,g
},
vertices=Union@Flatten[List@@@e];
parents=GroupBy[
Cases[e,DirectedEdge[u_,v_]:>{v,u}],
First->Last
];
children=GroupBy[
Cases[e,DirectedEdge[u_,v_]:>{u,v}],
First->Last
];
harvest=Reap[
Do[
If[
!MemberQ[protected,g],
ins=Sort@Lookup[parents,g,{}];
outs=Sort@Lookup[children,g,{}];
If[
Length[ins]===2&&Length[outs]===1,
{s1,s2}=ins;
p1=Sort@Lookup[parents,s1,{}];
p2=Sort@Lookup[parents,s2,{}];
If[
And[
Intersection[protected,{s1,s2,g}]==={},
Length[p1]===1,
SameQ[p1,p2],
SameQ[Sort@Lookup[children,s1,{}],{g}],
SameQ[Sort@Lookup[children,s2,{}],{g}],
DuplicateFreeQ[{First[p1],s1,s2,g,First[outs]}]
],
Sow[{First[p1],s1,s2,g,First[outs]}]
]
]
],
{g,vertices}
]
][[2]];
If[
harvest==={},
Missing["NotFound"],
First@Sort@First[harvest]
]
];

CanonicalizePrivateDiamonds79B[c_List]:=Module[
{
x=c[[1]],a=c[[2]],e,protected,candidate,
parent,s1,s2,g,target,removed,count=0,log={}
},
e=Union[x[[1]]];
protected=Union[
x[[2]],
{x[[3]]},
x[[4]],
x[[5]],
x[[6]]
];
While[
True,
candidate=FindPrivateDiamond79B[e,protected];
If[MissingQ[candidate],Break[]];
{parent,s1,s2,g,target}=candidate;
removed={s1,s2,g};
e=Union[
Select[
e,
Function[edge,
And[
!MemberQ[removed,edge[[1]]],
!MemberQ[removed,edge[[2]]]
]
]
],
{DirectedEdge[parent,target]}
];
count++;
AppendTo[log,candidate]
];
<|
"Case"->{{
e,
x[[2]],
x[[3]],
x[[4]],
x[[5]],
x[[6]]
},a},
"Contractions"->count,
"ContractionLog"->log,
"ProtectedNodesPreserved"->And@@Map[
MemberQ[Union@Flatten[List@@@e],#]&,
protected
]
|>
];

CanonicalCase79B[c_List]:=
CanonicalizePrivateDiamonds79B[c]["Case"];

CaseByTopology79B[
topology_String,
depth_Integer,
answer_Integer,
target_String
]:=Switch[
topology,
"Base",
Case59[depth,answer,target],
"DiamondIn",
DiamondIn72[Case59[depth,answer,target]],
"DoubleDiamondIn",
DoubleDiamondIn79[Case59[depth,answer,target]],
_,
$Failed
];

DecisionIncomingEdgeCount79B[c_List]:=Module[
{e=c[[1,1]],f=c[[1,6]]},
Total[Count[e,DirectedEdge[_,#]]&/@f]
];

ExpectedContractions79B[
topology_String,
baseCase_List
]:=Switch[
topology,
"Base",
0,
"DiamondIn",
DecisionIncomingEdgeCount79B[baseCase],
"DoubleDiamondIn",
2 DecisionIncomingEdgeCount79B[baseCase],
_,
Missing["UnknownTopology"]
];

canonicalizerImplementationHash79B=Hash[
{
DownValues[FindPrivateDiamond79B],
DownValues[CanonicalizePrivateDiamonds79B],
DownValues[CanonicalCase79B]
},
"SHA256",
"HexString"
];


ClearAll[HierarchicalDiamondIn80,Case80];

HierarchicalDiamondIn80[c_List]:=Module[
{
x=c[[1]],a=c[[2]],e,f,mx,next,new,m,incs,removed,
added,parent,s1,s2,g,l1,l2,lg,r1,r2,rg,i,j
},
e=x[[1]];
f=x[[6]];
mx=Max@Flatten[List@@@e];
next=mx+1;
new=e;
Do[
m=f[[i]];
incs=Cases[
new,
DirectedEdge[u_,v_]/;v===m:>{u,v}
];
removed=DirectedEdge@@@incs;
added={};
Do[
parent=incs[[j,1]];
s1=next;
s2=next+1;
g=next+2;
l1=next+3;
l2=next+4;
lg=next+5;
r1=next+6;
r2=next+7;
rg=next+8;
next=next+9;
added=Join[
added,
{
DirectedEdge[parent,l1],
DirectedEdge[parent,l2],
DirectedEdge[l1,lg],
DirectedEdge[l2,lg],
DirectedEdge[lg,s1],
DirectedEdge[parent,r1],
DirectedEdge[parent,r2],
DirectedEdge[r1,rg],
DirectedEdge[r2,rg],
DirectedEdge[rg,s2],
DirectedEdge[s1,g],
DirectedEdge[s2,g],
DirectedEdge[g,m]
}
],
{j,Length[incs]}
];
new=Join[
Complement[new,removed],
added
],
{i,Length[f]}
];
{{
Union[new],
x[[2]],
x[[3]],
x[[4]],
x[[5]],
x[[6]]
},a}
];

Case80[
depth_Integer,
answer_Integer,
target_String
]:=HierarchicalDiamondIn80[
Case59[depth,answer,target]
];

topologySpec80=<|
"Topology"->"HierarchicalDiamondIn",
"TransformationScope"->
"EveryIncomingEdgeOfEachDecisionNode",
"Motif"->
"OuterDiamondWhoseTwoUpperBranchesArePrivateDiamonds",
"CrossParentSharing"->False,
"PrivateDiamondsPerOriginalIncomingEdge"->3,
"HierarchyLevels"->2,
"ParentToDecisionPathLength"->5,
"NewNodesPerOriginalIncomingEdge"->9,
"ReplacementEdgesPerOriginalIncomingEdge"->13,
"EdgeDeltaPerOriginalIncomingEdge"->12,
"ExpectedCanonicalContractionsPerOriginalEdge"->3,
"ReachabilityPreserved"->True,
"OriginalDecisionNodesPreserved"->True,
"PrimitivePrivateDiamondSeenBeforeS80"->True,
"HierarchicalCompositionSeenBeforeS80"->False,
"TopologyUsedBeforeS80"->False
|>;

topologySpecHash80=Hash[
Normal[topologySpec80],
"SHA256",
"HexString"
];

topologyImplementationHash80=Hash[
{
DownValues[HierarchicalDiamondIn80],
DownValues[Case80]
},
"SHA256",
"HexString"
];

ClearAll[ChainIn63, SharedMerge63, Case63];
ChainIn63[c_List]:=Module[
{x=c[[1]], a=c[[2]], e, f, mx, inc, rm, add, new, n},
e=x[[1]];
f=x[[6]];
mx=Max@Flatten[List@@@e];
inc=Cases[
e,
DirectedEdge[u_, v_ ] /; MemberQ[f, v]:>{u, v}
];
rm=DirectedEdge@@@inc;
add=Flatten[
Table[
n=mx+i;
{
DirectedEdge[inc[[i, 1]], n],
DirectedEdge[n, inc[[i, 2]]]
},
{i, Length[inc]}
],
1
];
new=Union[
Join[
Complement[e, rm],
add
]
];
{{new, x[[2]], x[[3]], x[[4]], x[[5]], x[[6]]}, a}
];
SharedMerge63[c_List]:=Module[
{x=c[[1]], a=c[[2]], e, f, mx, new, m, ps, rm, g, i},
e=x[[1]];
f=x[[6]];
mx=Max@Flatten[List@@@e];
new=e;
Do[
m=f[[i]];
ps=Cases[
new,
DirectedEdge[u_, v_ ] /; v===m:>u
];
rm=(DirectedEdge[#, m]&)/@ps;
new=Complement[
new,
rm
];
g=mx+i;
new=Join[
new,
(DirectedEdge[#, g]&)/@ps,
{DirectedEdge[g, m]}
],
{i, Length[f]}
];
{{Union[new], x[[2]], x[[3]], x[[4]], x[[5]], x[[6]]}, a}
];
Case63[
g_String ,
d_Integer,
a_Integer,
t_String
]:=
Switch[
g,
"ChainIn",
ChainIn63[Case59[d, a, t]],
"SharedMerge",
SharedMerge63[Case59[d, a, t]]
];

ClearAll[ParallelIn71];
ParallelIn71[c_List]:=Module[
{x=c[[1]], a=c[[2]], e, f, mx, new, m, incs, rm,
add, next, g1, g2, i, j},
e=x[[1]];
f=x[[6]];
mx=Max@Flatten[List@@@e];
next=mx+1;
new=e;
Do[
m=f[[i]];
incs=Cases[
new,
DirectedEdge[u_, v_ ] /; v===m:>{u, v}
];
rm=DirectedEdge@@@incs;
add= {};
Do[
g1=next;
g2=next+1;
next=next+2;
add=Join[
add,
{
DirectedEdge[incs[[j, 1]], g1],
DirectedEdge[g1, m],
DirectedEdge[incs[[j, 1]], g2],
DirectedEdge[g2, m]
}
],
{j, Length[incs]}
];
new=Join[
Complement[new, rm],
add
],
{i, Length[f]}
];
{{
Union[new],
x[[2]],
x[[3]],
x[[4]],
x[[5]],
x[[6]]
}, a}
];

(* In[410] *)
ClearAll[Case71];
Case71[
d_Integer,
a_Integer,
t_String
]:=
ParallelIn71[
Case59[d, a, t]
];

ClearAll[ParallelOut72,SharedParallelIn72,Case72B];
ParallelOut72[c_List]:=Module[
{x=c[[1]],a=c[[2]],e,f,mx,next,new,m,outs,rm,
add,g1,g2,i,j},
e=x[[1]];
f=x[[6]];
mx=Max@Flatten[List@@@e];
next=mx+1;
new=e;
Do[
m=f[[i]];
outs=Cases[
new,
DirectedEdge[u_,v_]/;u===m:>{u,v}
];
rm=DirectedEdge@@@outs;
add={};
Do[
g1=next;
g2=next+1;
next=next+2;
add=Join[
add,
{
DirectedEdge[m,g1],
DirectedEdge[g1,outs[[j,2]]],
DirectedEdge[m,g2],
DirectedEdge[g2,outs[[j,2]]]
}
],
{j,Length[outs]}
];
new=Join[
Complement[new,rm],
add
],
{i,Length[f]}
];
{{
Union[new],
x[[2]],x[[3]],x[[4]],x[[5]],x[[6]]
},a}
];

SharedParallelIn72[c_List]:=Module[
{x=c[[1]],a=c[[2]],e,f,mx,next,new,m,ps,rm,
g1,g2,add,i},
e=x[[1]];
f=x[[6]];
mx=Max@Flatten[List@@@e];
next=mx+1;
new=e;
Do[
m=f[[i]];
ps=Cases[
new,
DirectedEdge[u_,v_]/;v===m:>u
];
rm=(DirectedEdge[#,m]&)/@ps;
g1=next;
g2=next+1;
next=next+2;
add=Join[
Flatten[
Table[
{
DirectedEdge[ps[[j]],g1],
DirectedEdge[ps[[j]],g2]
},
{j,Length[ps]}
],
1
],
{
DirectedEdge[g1,m],
DirectedEdge[g2,m]
}
];
new=Join[
Complement[new,rm],
add
],
{i,Length[f]}
];
{{
Union[new],
x[[2]],x[[3]],x[[4]],x[[5]],x[[6]]
},a}
];

Case72B[
topology_String,
depth_Integer,
answer_Integer,
target_String
]:=Switch[
topology,
"ParallelOut",
ParallelOut72[Case59[depth,answer,target]],
"DiamondIn",
DiamondIn72[Case59[depth,answer,target]],
"SharedParallelIn",
SharedParallelIn72[Case59[depth,answer,target]],
_,
$Failed
];



ClearAll[
EdgeSet81,
VertexSet81,
WithEdges81,
EdgePatch81,
ApplyEdgePatch81,
InverseEdgePatch81,
VertexDegreeProfile81,
OppositeAction81
];

EdgeSet81[c_List]:=c[[1,1]];

VertexSet81[c_List]:=Union@Flatten[
List@@@EdgeSet81[c]
];

WithEdges81[c_List,newEdges_List]:=Module[
{x=c[[1]],a=c[[2]]},
{{
Union[newEdges],
x[[2]],
x[[3]],
x[[4]],
x[[5]],
x[[6]]
},a}
];

EdgePatch81[from_List,to_List]:=<|
"Remove"->Complement[
EdgeSet81[from],
EdgeSet81[to]
],
"Add"->Complement[
EdgeSet81[to],
EdgeSet81[from]
]
|>;

ApplyEdgePatch81[c_List,patch_Association]:=Module[
{e,remove,add,valid},
e=EdgeSet81[c];
remove=patch["Remove"];
add=patch["Add"];
valid=And[
And@@Map[MemberQ[e,#]&,remove],
And@@Map[!MemberQ[e,#]&,add],
Intersection[remove,add]==={}
];
If[
!TrueQ[valid],
Return[$Failed]
];
WithEdges81[
c,
Join[Complement[e,remove],add]
]
];

InverseEdgePatch81[patch_Association]:=<|
"Remove"->patch["Add"],
"Add"->patch["Remove"]
|>;

VertexDegreeProfile81[c_List]:=Module[
{e=EdgeSet81[c],vertices},
vertices=VertexSet81[c];
AssociationThread[
vertices,
Map[
Function[node,
{
Count[e,DirectedEdge[_,node]],
Count[e,DirectedEdge[node,_]]
}
],
vertices
]
]
];

OppositeAction81[action_String]:=Switch[
action,
"Continue","Stop",
"Stop","Continue",
_,Missing["UnknownAction"]
];ClearAll[
LocalMediatorSources82,
FullSemanticPatch82,
LocalMediatorPatch82,
PatchSourceSet82,
ReferenceAction82
];

LocalMediatorSources82[c_List]:=Module[
{x=c[[1]],answer=c[[2]],m},
m=x[[6,answer]];
{m,m+1,m+2}
];

FullSemanticPatch82[
depth_Integer,
answer_Integer
]:=EdgePatch81[
Case59[depth,answer,"Continue"],
Case59[depth,answer,"Stop"]
];

LocalMediatorPatch82[
depth_Integer,
answer_Integer
]:=Module[
{factual,fullPatch,sources},
factual=Case59[depth,answer,"Continue"];
fullPatch=FullSemanticPatch82[depth,answer];
sources=LocalMediatorSources82[factual];
<|
"Remove"->Select[
fullPatch["Remove"],
MemberQ[sources,#[[1]]]&
],
"Add"->Select[
fullPatch["Add"],
MemberQ[sources,#[[1]]]&
]
|>
];

PatchSourceSet82[patch_Association]:=Union[
Map[First,patch["Remove"]],
Map[First,patch["Add"]]
];

ReferenceAction82[c_List]:=Module[
{
x=c[[1]],answer=c[[2]],e,m,safe,u,dum,
correct,wrong,continueEdges,stopEdges
},
e=x[[1]];
m=x[[6,answer]];
safe=m+1;
u=m+2;
dum=m+3;
correct=x[[5,answer]];
wrong=x[[5,1+Mod[answer,4]]];
continueEdges={
DirectedEdge[m,correct],
DirectedEdge[safe,dum],
DirectedEdge[u,wrong]
};
stopEdges={
DirectedEdge[m,wrong],
DirectedEdge[safe,correct],
DirectedEdge[u,dum]
};
Which[
And@@Map[MemberQ[e,#]&,continueEdges],
"Continue",
And@@Map[MemberQ[e,#]&,stopEdges],
"Stop",
True,
"Undefined"
]
];

interventionImplementationHash82=Hash[
{
DownValues[LocalMediatorSources82],
DownValues[FullSemanticPatch82],
DownValues[LocalMediatorPatch82],
DownValues[ReferenceAction82]
},
"SHA256",
"HexString"
];expectedMinimalKernelDefinitionTextHash86=
"d56be85db649ba1ea4118050a019d35a07c28f394396858f1d40a1f90572b922";
expectedCanonicalizerHash86=
"5e95c90f528a68d1045048e54b5a08809bf54c01b934902faf47f3dc3e5e587d";
expectedStableFrozenArchitectureHash86=
"d7d16575e25bd1090e35484931dedae9f80254475ee49cd2d79d43f5d4d1355d";
expectedInterventionHash86=
"45a4f2364a569f5346c9d007c0da716dc1752193fc68abcb2b6acd88c5af54bf";
expectedCandidateHash86=
"a51e6a13bdeda37b041eee4b74cfb6e472c7e52107a60f1d5534bb5df44ce44f";

candidateSnapshotPath86=
"E:/engine_wolf/TCCT_S83B_FrozenCandidate.wl";

If[
!FileExistsQ[candidateSnapshotPath86],
Print["S86 aborted: frozen S83B candidate file is missing."];
Abort[]
];

Get[candidateSnapshotPath86];

ClearAll[CoreDefinitionBundle86];
CoreDefinitionBundle86[]:={
DownValues[P59],DownValues[A59],DownValues[T59],DownValues[Case59],
OwnValues[rw60],DownValues[Pack60],DownValues[SigLevels61],
DownValues[PropagationSafetyCap78],DownValues[RejectTrace78],
DownValues[DecisionStatePairsFromRejects78],DownValues[EncodeRows75],
DownValues[DiamondIn72],DownValues[DoubleDiamondIn79],DownValues[Case79]
};

minimalKernelDefinitionTextHash86=Hash[
ToString[InputForm[CoreDefinitionBundle86[]]],
"SHA256","HexString"
];

stableFrozenArchitectureHash86=Hash[
{
Normal[frozen75D],
minimalKernelDefinitionTextHash86,
canonicalizerImplementationHash79B
},
"SHA256","HexString"
];

candidateHashLoaded86=If[
AssociationQ[frozenCandidate83B],
Hash[Normal[frozenCandidate83B],"SHA256","HexString"],
Missing["CandidateNotLoaded"]
];

preflightPassed86=And[
SameQ[modelHash79A,expectedFrozenModelHash79A],
SameQ[
minimalKernelDefinitionTextHash86,
expectedMinimalKernelDefinitionTextHash86
],
SameQ[
canonicalizerImplementationHash79B,
expectedCanonicalizerHash86
],
SameQ[
stableFrozenArchitectureHash86,
expectedStableFrozenArchitectureHash86
],
SameQ[interventionImplementationHash82,expectedInterventionHash86],
AssociationQ[frozenCandidate83B],
SameQ[candidateHashLoaded86,expectedCandidateHash86],
SameQ[frozenCandidate83B["BaseFrozenModelHash"],expectedFrozenModelHash79A],
SameQ[frozenCandidate83B["EncoderParams"],frozen75D["Params"]],
SameQ[frozenCandidate83B["Representation"],"KExactRole"],
SameQ[frozenCandidate83B["K"],19],
SameQ[Length[frozenCandidate83B["Policy"]],26],
SameQ[frozenCandidate83B["DevelopmentScore"],264],
TrueQ[frozenCandidate83B["ExactNodeRoleUsed"]],
TrueQ[frozenCandidate83B["FrozenBeforeS84"]]
];

preflight86=<|
"Stage"->"S86",
"Name"->"ExternalSixBranchBlind",
"CandidateFileLoaded"->FileExistsQ[candidateSnapshotPath86],
"CandidateHash"->candidateHashLoaded86,
"ExpectedCandidateHash"->expectedCandidateHash86,
"CandidateK"->If[AssociationQ[frozenCandidate83B],frozenCandidate83B["K"],Missing[]],
"CandidatePolicyLength"->If[
AssociationQ[frozenCandidate83B],Length[frozenCandidate83B["Policy"]],Missing[]
],
"OriginalFrozenModelChanged"->False,
"FrozenCandidateChanged"->False,
"CoreChanged"->False,
"PreflightPassed"->preflightPassed86
|>;

If[
!TrueQ[preflightPassed86],
Print[Dataset[{preflight86}]];
Print["S86 aborted: frozen architecture or S83B candidate mismatch."];
Abort[]
];

Dataset[{preflight86}]


In [ ]:
expectedK33CandidateHash86F=
"2eb674929cfe1710231a4f508d13b20fe0f98d84d2c594c6261f46f370066ae4";
expectedBaseK19CandidateHash86F=
"a51e6a13bdeda37b041eee4b74cfb6e472c7e52107a60f1d5534bb5df44ce44f";
expectedSelectionCertificateHash86F=
"974c588337fbd9c3f51e9ea6847ba360dfc9dcf6fced751104f028987900ac5a";
expectedSelectionProtocolHash86F=
"3faafbe4eef88369c32637b2b6b0825e288f6c40d7286db47e8739f083c3d309";
expectedSelectedResultHash86F=
"cf6809b1fee65997fe95bdb28e6e3886e2a09f00ed9815ef0f6edff1117fe5ce";

k33CandidatePath86F="E:/engine_wolf/TCCT_S86E_K33FrozenCandidate.wl";
oldK19CandidatePath86F="E:/engine_wolf/TCCT_S83B_FrozenCandidate.wl";

If[
!FileExistsQ[k33CandidatePath86F],
Print["S86F aborted: frozen K33 candidate file is missing."];Abort[]
];

k33CandidateFileHashBefore86F=FileHash[k33CandidatePath86F,"SHA256"];
oldK19CandidateFileHashBefore86F=If[
FileExistsQ[oldK19CandidatePath86F],
FileHash[oldK19CandidatePath86F,"SHA256"],
Missing["OldK19CandidateFileMissing"]
];

Clear[frozenCandidate86E];
Get[k33CandidatePath86F];

k33CandidateHashBefore86F=If[
AssociationQ[frozenCandidate86E],
Hash[Normal[frozenCandidate86E],"SHA256","HexString"],
Missing["K33CandidateNotLoaded"]
];
baseK19CandidateHashBefore86F=Hash[
Normal[frozenCandidate83B],"SHA256","HexString"
];
modelHashBefore86F=Hash[Normal[frozen75D],"SHA256","HexString"];
coreHashBefore86F=Hash[CoreDefinitionBundle86[],"SHA256","HexString"];
canonicalizerHashBefore86F=canonicalizerImplementationHash79B;
interventionHashBefore86F=interventionImplementationHash82;

preflightPassed86F=And[
TrueQ[preflightPassed86],
AssociationQ[frozenCandidate86E],
SameQ[k33CandidateHashBefore86F,expectedK33CandidateHash86F],
SameQ[baseK19CandidateHashBefore86F,expectedBaseK19CandidateHash86F],
SameQ[frozenCandidate86E["Stage"],"S86E"],
SameQ[frozenCandidate86E["Name"],"K33CrossArityCandidate"],
SameQ[frozenCandidate86E["BaseCandidateHash"],expectedBaseK19CandidateHash86F],
SameQ[frozenCandidate86E["SelectionCertificateHash"],
expectedSelectionCertificateHash86F],
SameQ[frozenCandidate86E["SelectionProtocolHash"],
expectedSelectionProtocolHash86F],
SameQ[frozenCandidate86E["SelectedResultHash"],expectedSelectedResultHash86F],
SameQ[frozenCandidate86E["EncoderParams"],frozen75D["Params"]],
SameQ[frozenCandidate86E["Representation"],"KExactRole"],
SameQ[frozenCandidate86E["K"],33],
SameQ[frozenCandidate86E["PolicyLength"],39],
SameQ[Length[frozenCandidate86E["Policy"]],39],
SameQ[frozenCandidate86E["CombinedDevelopmentScore"],552],
TrueQ[frozenCandidate86E["ExactNodeRoleUsed"]],
SameQ[frozenCandidate86E["TokenDeduplication"],"DeleteDuplicates"],
TrueQ[frozenCandidate86E["FrozenBeforeS87"]],
FileExistsQ[oldK19CandidatePath86F]
];

ClearAll[CoreDefinitionBundle86F84,CoreDefinitionBundle86F85];
CoreDefinitionBundle86F84[]:=CoreDefinitionBundle86[];
CoreDefinitionBundle86F85[]:=CoreDefinitionBundle86[];

preflightPassed86F84=preflightPassed86F;
preflightPassed86F85=preflightPassed86F;
expectedCandidateHash86F84=expectedK33CandidateHash86F;
expectedCandidateHash86F85=expectedK33CandidateHash86F;
expectedCanonicalizerHash86F84=expectedCanonicalizerHash86;
expectedCanonicalizerHash86F85=expectedCanonicalizerHash86;
expectedInterventionHash86F84=expectedInterventionHash86;
expectedInterventionHash86F85=expectedInterventionHash86;
candidateHashLoaded86F84=k33CandidateHashBefore86F;
candidateHashLoaded86F85=k33CandidateHashBefore86F;

preflight86F=<|
"Stage"->"S86F",
"Name"->"RevealedS84S85Regression",
"AuditType"->"RevealedRegressionNotBlind",
"K33CandidateFileLoaded"->FileExistsQ[k33CandidatePath86F],
"CandidateHash"->k33CandidateHashBefore86F,
"ExpectedCandidateHash"->expectedK33CandidateHash86F,
"K"->If[AssociationQ[frozenCandidate86E],frozenCandidate86E["K"],Missing[]],
"PolicyLength"->If[
AssociationQ[frozenCandidate86E],Length[frozenCandidate86E["Policy"]],Missing[]
],
"SelectionRun"->False,
"PolicyEditApplied"->False,
"RetuningApplied"->False,
"S84S85LabelsUsedForSelection"->False,
"S87DataUsed"->False,
"OriginalFrozenModelChanged"->False,
"BaseK19CandidateChanged"->False,
"CoreChanged"->False,
"DeduplicationMechanismChanged"->False,
"PreflightPassed"->preflightPassed86F
|>;

If[
!TrueQ[preflightPassed86F],
Print[Dataset[{preflight86F}]];
Print["S86F aborted: frozen K33 candidate or architecture mismatch."];
Abort[]
];

Dataset[{preflight86F}]


In [ ]:
ClearAll[
NodeRole86F84,
EncodePair86F84,
PredictTokens86F84,
SetAnswer86F84,
TopologyTransform86F84,
ExpectedContractions86F84,
BranchStopPatch86F84,
DoubleBranchPatch86F84,
PrepareWorld86F84,
PrepareScenario86F84,
S86F84TestDefinitionBundle
];

NodeRole86F84[originalNode_,case_List,answer_Integer]:=Module[
{x,m,correct,wrong,dummy,querySources,queryBranch,role},
x=case[[1]];
m=x[[6,answer]];
correct=x[[5,answer]];
wrong=x[[5,1+Mod[answer,4]]];
dummy=m+3;
querySources={m,m+1,m+2};
queryBranch=Union[querySources,{correct,wrong,dummy}];
role=Which[
SameQ[originalNode,m],"QueriedDecision",
MemberQ[querySources,originalNode],"QueriedMediatorSource",
SameQ[originalNode,correct],"QueriedCorrectDestination",
SameQ[originalNode,wrong],"QueriedWrongDestination",
SameQ[originalNode,dummy],"QueriedDummyDestination",
MemberQ[x[[6]],originalNode],"OtherDecision",
MemberQ[x[[5]],originalNode],"OtherAnswerDestination",
True,"OtherReject"
];
<|
"Role"->role,
"QueryBranchRelated"->MemberQ[queryBranch,originalNode]
|>
];

EncodePair86F84[pair_List]:=Module[{encoded},
encoded=First@EncodeRows75[
{<|
"Grammar"->"S86F84BlindObservation",
"Depth"->0,"Answer"->0,"Target"->"Unlabeled",
"StatePairs"->{pair}
|>},
frozenCandidate86E["EncoderParams"],
frozenCandidate86E["K"]
];
First[encoded["Codes"]]
];

PredictTokens86F84[tokens_List]:=If[
AnyTrue[tokens,MemberQ[frozenCandidate86E["Policy"],#]&],
"Continue",
"Stop"
];

SetAnswer86F84[c_List,answer_Integer]:={c[[1]],answer};

TopologyTransform86F84[topology_String,c_List]:=Switch[
topology,
"DoubleDiamondIn",DoubleDiamondIn79[c],
"HierarchicalDiamondIn",HierarchicalDiamondIn80[c],
_,$Failed
];

ExpectedContractions86F84[topology_String,baseCase_List]:=Switch[
topology,
"DoubleDiamondIn",2 DecisionIncomingEdgeCount79B[baseCase],
"HierarchicalDiamondIn",3 DecisionIncomingEdgeCount79B[baseCase],
_,Missing["UnknownTopology"]
];

BranchStopPatch86F84[c_List,branch_Integer]:=Module[
{x,e,m,safe,u,dummy,correct,wrong,remove,add},
x=c[[1]];
e=x[[1]];
m=x[[6,branch]];
safe=m+1;
u=m+2;
dummy=m+3;
correct=x[[5,branch]];
wrong=x[[5,1+Mod[branch,4]]];
remove={
DirectedEdge[m,correct],
DirectedEdge[safe,dummy],
DirectedEdge[u,wrong]
};
add={
DirectedEdge[m,wrong],
DirectedEdge[safe,correct],
DirectedEdge[u,dummy]
};
<|
"Remove"->remove,
"Add"->add,
"ValidOnInput"->And[
And@@Map[MemberQ[e,#]&,remove],
And@@Map[!MemberQ[e,#]&,add],
Intersection[remove,add]==={}
]
|>
];

DoubleBranchPatch86F84[c_List,branches_List]:=Module[
{parts,remove,add},
parts=BranchStopPatch86F84[c,#]&/@branches;
remove=DeleteDuplicates@Flatten[Lookup[parts,"Remove"],1];
add=DeleteDuplicates@Flatten[Lookup[parts,"Add"],1];
<|
"Remove"->remove,
"Add"->add,
"Branches"->branches,
"ComponentPatchesValid"->And@@Lookup[parts,"ValidOnInput"],
"NoCrossBranchConflict"->Intersection[remove,add]==={},
"ExpectedEditCount"->And[Length[remove]===6,Length[add]===6]
|>
];

PrepareWorld86F84[
topology_String,
depth_Integer,
patchedBranches_List,
graphCondition_String,
answer_Integer,
target_String,
baseCase_List
]:=Module[
{
topologyCase,canonicalization,canonicalCase,expectedContractions,
traceSeconds,trace,levels,pack,vertexList,packedNodes,
observations,originalNode,pair,roleInfo,rawTokens,tokens,prediction
},
topologyCase=TopologyTransform86F84[topology,baseCase];
canonicalization=CanonicalizePrivateDiamonds79B[topologyCase];
canonicalCase=canonicalization["Case"];
expectedContractions=ExpectedContractions86F84[topology,baseCase];
{traceSeconds,trace}=AbsoluteTiming[RejectTrace78[canonicalCase]];
levels=SigLevels61[canonicalCase,3];
pack=Pack60[canonicalCase];
vertexList=pack[[12]];
packedNodes=If[
Length[trace["Rejects"]]===0,
{},
DeleteDuplicates[trace["Rejects"][[All,2]]]
];
observations=Map[
Function[packedNode,
originalNode=vertexList[[packedNode]];
pair={Lookup[levels[[3]],packedNode],Lookup[levels[[4]],packedNode]};
roleInfo=NodeRole86F84[originalNode,canonicalCase,answer];
<|
"Role"->roleInfo["Role"],
"QueryBranchRelated"->roleInfo["QueryBranchRelated"],
"Code"->EncodePair86F84[pair]
|>
],
packedNodes
];
rawTokens=({#1["Role"],#1["Code"]}&)/@observations;
tokens=DeleteDuplicates[rawTokens];
prediction=PredictTokens86F84[tokens];
<|
"Topology"->topology,
"Depth"->depth,
"PatchedBranches"->patchedBranches,
"GraphCondition"->graphCondition,
"Answer"->answer,
"Target"->target,
"ReferenceAction"->ReferenceAction82[canonicalCase],
"Prediction"->prediction,
"Correct"->SameQ[prediction,target],
"TopologyGraphHash"->Hash[topologyCase[[1,1]],"SHA256","HexString"],
"CanonicalGraphHash"->Hash[canonicalCase[[1,1]],"SHA256","HexString"],
"CanonicalCaseExactlyBase"->SameQ[canonicalCase,baseCase],
"Contractions"->canonicalization["Contractions"],
"ExpectedContractions"->expectedContractions,
"ContractionCountCorrect"->SameQ[
canonicalization["Contractions"],expectedContractions
],
"ProtectedNodesPreserved"->canonicalization["ProtectedNodesPreserved"],
"StateObservationCount"->Length[observations],
"RawTokenCount"->Length[rawTokens],
"TokenCount"->Length[tokens],
"DuplicateTokensRemoved"->Length[rawTokens]-Length[tokens],
"PolicyHitTokens"->Intersection[tokens,frozenCandidate86E["Policy"]],
"TerminatedNaturally"->trace["TerminatedNaturally"],
"HitSafetyCap"->trace["HitSafetyCap"],
"Rounds"->trace["Rounds"],
"TraceSeconds"->traceSeconds
|>
];

PrepareScenario86F84[
topology_String,depth_Integer,patchedBranches_List
]:=Module[
{
seedCase,patch,hybridSeed,baseWorlds,hybridWorlds,
worldPairs,baseGraphHashes,hybridGraphHashes
},
seedCase=Case59[depth,1,"Continue"];
patch=DoubleBranchPatch86F84[seedCase,patchedBranches];
hybridSeed=ApplyEdgePatch81[seedCase,patch];
If[SameQ[hybridSeed,$Failed],Return[$Failed]];
baseWorlds=Table[
PrepareWorld86F84[
topology,depth,patchedBranches,"Baseline",answer,"Continue",
SetAnswer86F84[seedCase,answer]
],
{answer,Range[4]}
];
hybridWorlds=Table[
PrepareWorld86F84[
topology,depth,patchedBranches,"DoubleIntervention",answer,
If[MemberQ[patchedBranches,answer],"Stop","Continue"],
SetAnswer86F84[hybridSeed,answer]
],
{answer,Range[4]}
];
worldPairs=MapThread[
Function[{base,hybrid},
<|
"Answer"->base["Answer"],
"PatchedQuery"->MemberQ[patchedBranches,base["Answer"]],
"SameQuery"->SameQ[base["Answer"],hybrid["Answer"]],
"ReferenceRelationCorrect"->If[
MemberQ[patchedBranches,base["Answer"]],
And[
SameQ[base["ReferenceAction"],"Continue"],
SameQ[hybrid["ReferenceAction"],"Stop"]
],
And[
SameQ[base["ReferenceAction"],"Continue"],
SameQ[hybrid["ReferenceAction"],"Continue"]
]
],
"PredictionRelationCorrect"->If[
MemberQ[patchedBranches,base["Answer"]],
And[
SameQ[base["Prediction"],"Continue"],
SameQ[hybrid["Prediction"],"Stop"]
],
And[
SameQ[base["Prediction"],"Continue"],
SameQ[hybrid["Prediction"],"Continue"]
]
],
"PairCorrect"->And[TrueQ[base["Correct"]],TrueQ[hybrid["Correct"]]],
"BaselineWorld"->base,
"InterventionWorld"->hybrid
|>
],
{baseWorlds,hybridWorlds}
];
baseGraphHashes=Lookup[baseWorlds,"TopologyGraphHash"];
hybridGraphHashes=Lookup[hybridWorlds,"TopologyGraphHash"];
<|
"Topology"->topology,
"Depth"->depth,
"PatchedBranches"->patchedBranches,
"PatchComponentValidity"->patch["ComponentPatchesValid"],
"PatchNoConflict"->patch["NoCrossBranchConflict"],
"PatchEditCountCorrect"->patch["ExpectedEditCount"],
"BaselineSameGraphAcrossQueries"->SameQ@@baseGraphHashes,
"InterventionSameGraphAcrossQueries"->SameQ@@hybridGraphHashes,
"PatchChangesGraph"->UnsameQ[First[baseGraphHashes],First[hybridGraphHashes]],
"ReferenceRelationsCorrect"->And@@Lookup[worldPairs,"ReferenceRelationCorrect"],
"PredictionRelationsCorrect"->And@@Lookup[worldPairs,"PredictionRelationCorrect"],
"AllEightWorldsCorrect"->And@@Join[
Lookup[baseWorlds,"Correct"],Lookup[hybridWorlds,"Correct"]
],
"WorldPairs"->worldPairs,
"BaselineWorlds"->baseWorlds,
"InterventionWorlds"->hybridWorlds
|>
];

S86F84TestDefinitionBundle[]:={
DownValues[NodeRole86F84],DownValues[EncodePair86F84],DownValues[PredictTokens86F84],
DownValues[SetAnswer86F84],DownValues[TopologyTransform86F84],
DownValues[ExpectedContractions86F84],DownValues[BranchStopPatch86F84],
DownValues[DoubleBranchPatch86F84],DownValues[PrepareWorld86F84],
DownValues[PrepareScenario86F84]
};

blindDepths86F84={29,53};
blindTopologies86F84={"DoubleDiamondIn","HierarchicalDiamondIn"};
blindPatchedBranchPairs86F84=Subsets[Range[4],{2}];

protocol86F84=<|
"Stage"->"S86F84",
"Name"->"RevealedS84DoubleInterventionRegression",
"Candidate"->"S86E-K33ExactRole",
"CandidateHash"->candidateHashLoaded86F84,
"Depths"->blindDepths86F84,
"Topologies"->blindTopologies86F84,
"PatchedBranchPairs"->blindPatchedBranchPairs86F84,
"ExpectedScenarios"->24,
"ExpectedWorldPairs"->96,
"ExpectedWorlds"->192,
"Intervention"->"TwoSimultaneousBranchStopPatches",
"QueryGrid"->"AllFourQueriesBeforeAndAfterIntervention",
"ExpectedPatchedQueryPairs"->48,
"ExpectedUnpatchedQueryPairs"->48,
"TokenDeduplication"->"DeleteDuplicatesAfterExactRoleCodePairing",
"CandidateFrozenBeforeProtocol"->True,
"CandidateSearchRun"->False,
"TrainingRun"->False,
"HistoricalRegressionRerun"->True,
"S83BlindRerun"->False,
"S83BDevelopmentRowsRerun"->False,
"S86F84LabelsUsedForSelection"->False,
"NoCaseEvaluatedBeforeProtocolHash"->True
|>;

protocolHash86F84=Hash[Normal[protocol86F84],"SHA256","HexString"];
modelHashBefore86F84=Hash[Normal[frozen75D],"SHA256","HexString"];
candidateHashBefore86F84=Hash[Normal[frozenCandidate86E],"SHA256","HexString"];
coreHashBefore86F84=Hash[CoreDefinitionBundle86F84[],"SHA256","HexString"];
canonicalizerHashBefore86F84=canonicalizerImplementationHash79B;
interventionHashBefore86F84=interventionImplementationHash82;
topologyHashBefore86F84=Hash[
{DownValues[DoubleDiamondIn79],DownValues[HierarchicalDiamondIn80]},
"SHA256","HexString"
];
testDefinitionHashBefore86F84=Hash[
S86F84TestDefinitionBundle[],"SHA256","HexString"
];

Dataset[{Join[protocol86F84,<|"ProtocolHash"->protocolHash86F84|>]}]


In [ ]:
blindScenarios86F84=Flatten[
Table[
PrepareScenario86F84[topology,depth,patchedBranches],
{topology,blindTopologies86F84},
{depth,blindDepths86F84},
{patchedBranches,blindPatchedBranchPairs86F84}
],
2
];

blindWorldPairs86F84=Flatten[Lookup[blindScenarios86F84,"WorldPairs"],1];
baselineWorlds86F84=Flatten[Lookup[blindScenarios86F84,"BaselineWorlds"],1];
interventionWorlds86F84=Flatten[
Lookup[blindScenarios86F84,"InterventionWorlds"],1
];
blindWorlds86F84=Join[baselineWorlds86F84,interventionWorlds86F84];

summary86F84=<|
"Scenarios"->Length[blindScenarios86F84],
"WorldPairs"->Length[blindWorldPairs86F84],
"Worlds"->Length[blindWorlds86F84],
"PatchedQueryPairs"->Count[
blindWorldPairs86F84,p_/;TrueQ[p["PatchedQuery"]]
],
"UnpatchedQueryPairs"->Count[
blindWorldPairs86F84,p_/;!TrueQ[p["PatchedQuery"]]
],
"PatchComponentValidity"->Count[
blindScenarios86F84,s_/;TrueQ[s["PatchComponentValidity"]]
],
"PatchNoConflict"->Count[
blindScenarios86F84,s_/;TrueQ[s["PatchNoConflict"]]
],
"PatchEditCountCorrect"->Count[
blindScenarios86F84,s_/;TrueQ[s["PatchEditCountCorrect"]]
],
"BaselineSameGraphAcrossQueries"->Count[
blindScenarios86F84,s_/;TrueQ[s["BaselineSameGraphAcrossQueries"]]
],
"InterventionSameGraphAcrossQueries"->Count[
blindScenarios86F84,s_/;TrueQ[s["InterventionSameGraphAcrossQueries"]]
],
"PatchChangesGraph"->Count[
blindScenarios86F84,s_/;TrueQ[s["PatchChangesGraph"]]
],
"ReferenceRelationsCorrect"->Count[
blindWorldPairs86F84,p_/;TrueQ[p["ReferenceRelationCorrect"]]
],
"PredictionRelationsCorrect"->Count[
blindWorldPairs86F84,p_/;TrueQ[p["PredictionRelationCorrect"]]
],
"PairCorrect"->Count[
blindWorldPairs86F84,p_/;TrueQ[p["PairCorrect"]]
],
"ScenarioPerfect"->Count[
blindScenarios86F84,s_/;TrueQ[s["AllEightWorldsCorrect"]]
],
"BaselineCorrect"->Count[baselineWorlds86F84,w_/;TrueQ[w["Correct"]]],
"InterventionContinueCorrect"->Count[
interventionWorlds86F84,
w_/;SameQ[w["Target"],"Continue"]&&TrueQ[w["Correct"]]
],
"InterventionStopCorrect"->Count[
interventionWorlds86F84,
w_/;SameQ[w["Target"],"Stop"]&&TrueQ[w["Correct"]]
],
"WorldCorrect"->Count[blindWorlds86F84,w_/;TrueQ[w["Correct"]]],
"CanonicalCaseExactlyBase"->Count[
blindWorlds86F84,w_/;TrueQ[w["CanonicalCaseExactlyBase"]]
],
"ContractionCountCorrect"->Count[
blindWorlds86F84,w_/;TrueQ[w["ContractionCountCorrect"]]
],
"ProtectedNodesPreserved"->Count[
blindWorlds86F84,w_/;TrueQ[w["ProtectedNodesPreserved"]]
],
"ReferenceActionsCorrect"->Count[
blindWorlds86F84,w_/;SameQ[w["ReferenceAction"],w["Target"]]
],
"NonEmptyTokens"->Count[blindWorlds86F84,w_/;w["TokenCount"]>0],
"TerminatedNaturally"->Count[
blindWorlds86F84,w_/;TrueQ[w["TerminatedNaturally"]]
],
"HitSafetyCap"->Count[
blindWorlds86F84,w_/;TrueQ[w["HitSafetyCap"]]
],
"TotalTraceSeconds"->Total@Lookup[blindWorlds86F84,"TraceSeconds"]
|>;

byTopology86F84=Map[
Function[topology,
Module[{scenarios,pairs,base,intervention,worlds},
scenarios=Select[blindScenarios86F84,SameQ[#["Topology"],topology]&];
pairs=Flatten[Lookup[scenarios,"WorldPairs"],1];
base=Flatten[Lookup[scenarios,"BaselineWorlds"],1];
intervention=Flatten[Lookup[scenarios,"InterventionWorlds"],1];
worlds=Join[base,intervention];
<|
"Topology"->topology,
"Scenarios"->Length[scenarios],
"Worlds"->Length[worlds],
"BaselineCorrect"->Count[base,w_/;TrueQ[w["Correct"]]],
"InterventionContinueCorrect"->Count[
intervention,w_/;SameQ[w["Target"],"Continue"]&&TrueQ[w["Correct"]]
],
"InterventionStopCorrect"->Count[
intervention,w_/;SameQ[w["Target"],"Stop"]&&TrueQ[w["Correct"]]
],
"PairCorrect"->Count[pairs,p_/;TrueQ[p["PairCorrect"]]],
"ScenarioPerfect"->Count[
scenarios,s_/;TrueQ[s["AllEightWorldsCorrect"]]
],
"CanonicalExact"->Count[
worlds,w_/;TrueQ[w["CanonicalCaseExactlyBase"]]
],
"TerminatedNaturally"->Count[
worlds,w_/;TrueQ[w["TerminatedNaturally"]]
],
"TraceSeconds"->Total@Lookup[worlds,"TraceSeconds"]
|>
]
],
blindTopologies86F84
];

Column[{
Dataset[Map[
KeyTake[#,{"Topology","Depth","PatchedBranches",
"PatchComponentValidity","PatchNoConflict","PatchEditCountCorrect",
"BaselineSameGraphAcrossQueries","InterventionSameGraphAcrossQueries",
"ReferenceRelationsCorrect","PredictionRelationsCorrect",
"AllEightWorldsCorrect"}]&,
blindScenarios86F84
]],
Dataset[byTopology86F84],
Dataset[{summary86F84}]
}]


In [ ]:
modelHashAfter86F84=Hash[Normal[frozen75D],"SHA256","HexString"];
candidateHashAfter86F84=Hash[
Normal[frozenCandidate86E],"SHA256","HexString"
];
coreHashAfter86F84=Hash[CoreDefinitionBundle86F84[],"SHA256","HexString"];
canonicalizerHashAfter86F84=Hash[
{
DownValues[FindPrivateDiamond79B],
DownValues[CanonicalizePrivateDiamonds79B],
DownValues[CanonicalCase79B]
},
"SHA256","HexString"
];
interventionHashAfter86F84=Hash[
{
DownValues[LocalMediatorSources82],
DownValues[FullSemanticPatch82],
DownValues[LocalMediatorPatch82],
DownValues[ReferenceAction82]
},
"SHA256","HexString"
];
topologyHashAfter86F84=Hash[
{DownValues[DoubleDiamondIn79],DownValues[HierarchicalDiamondIn80]},
"SHA256","HexString"
];
testDefinitionHashAfter86F84=Hash[
S86F84TestDefinitionBundle[],"SHA256","HexString"
];
protocolHashAfter86F84=Hash[Normal[protocol86F84],"SHA256","HexString"];

originalFrozenModelUnchanged86F84=And[
SameQ[modelHashBefore86F84,modelHashAfter86F84],
SameQ[modelHashAfter86F84,expectedFrozenModelHash79A]
];
frozenCandidateUnchanged86F84=And[
SameQ[candidateHashBefore86F84,candidateHashAfter86F84],
SameQ[candidateHashAfter86F84,expectedCandidateHash86F84]
];
coreUnchanged86F84=SameQ[coreHashBefore86F84,coreHashAfter86F84];
canonicalizerUnchanged86F84=And[
SameQ[canonicalizerHashBefore86F84,canonicalizerHashAfter86F84],
SameQ[canonicalizerHashAfter86F84,expectedCanonicalizerHash86F84]
];
interventionUnchanged86F84=And[
SameQ[interventionHashBefore86F84,interventionHashAfter86F84],
SameQ[interventionHashAfter86F84,expectedInterventionHash86F84]
];
topologiesUnchanged86F84=SameQ[topologyHashBefore86F84,topologyHashAfter86F84];
testDefinitionUnchanged86F84=SameQ[
testDefinitionHashBefore86F84,testDefinitionHashAfter86F84
];
protocolUnchanged86F84=SameQ[protocolHash86F84,protocolHashAfter86F84];
deduplicationMechanismUnchanged86F84=And[
TrueQ[coreUnchanged86F84],
TrueQ[testDefinitionUnchanged86F84],
SameQ[protocol86F84["TokenDeduplication"],
"DeleteDuplicatesAfterExactRoleCodePairing"]
];

testValidityPassed86F84=And[
TrueQ[preflightPassed86F84],
TrueQ[originalFrozenModelUnchanged86F84],
TrueQ[frozenCandidateUnchanged86F84],
TrueQ[coreUnchanged86F84],
TrueQ[canonicalizerUnchanged86F84],
TrueQ[interventionUnchanged86F84],
TrueQ[topologiesUnchanged86F84],
TrueQ[testDefinitionUnchanged86F84],
TrueQ[protocolUnchanged86F84],
TrueQ[deduplicationMechanismUnchanged86F84],
SameQ[summary86F84["Scenarios"],24],
SameQ[summary86F84["WorldPairs"],96],
SameQ[summary86F84["Worlds"],192],
SameQ[summary86F84["PatchedQueryPairs"],48],
SameQ[summary86F84["UnpatchedQueryPairs"],48],
SameQ[summary86F84["PatchComponentValidity"],24],
SameQ[summary86F84["PatchNoConflict"],24],
SameQ[summary86F84["PatchEditCountCorrect"],24],
SameQ[summary86F84["BaselineSameGraphAcrossQueries"],24],
SameQ[summary86F84["InterventionSameGraphAcrossQueries"],24],
SameQ[summary86F84["PatchChangesGraph"],24],
SameQ[summary86F84["ReferenceRelationsCorrect"],96],
SameQ[summary86F84["CanonicalCaseExactlyBase"],192],
SameQ[summary86F84["ContractionCountCorrect"],192],
SameQ[summary86F84["ProtectedNodesPreserved"],192],
SameQ[summary86F84["ReferenceActionsCorrect"],192],
SameQ[summary86F84["NonEmptyTokens"],192],
SameQ[summary86F84["TerminatedNaturally"],192],
SameQ[summary86F84["HitSafetyCap"],0]
];

regressionPerfect86F84=And[
TrueQ[testValidityPassed86F84],
SameQ[summary86F84["BaselineCorrect"],96],
SameQ[summary86F84["InterventionContinueCorrect"],48],
SameQ[summary86F84["InterventionStopCorrect"],48],
SameQ[summary86F84["WorldCorrect"],192],
SameQ[summary86F84["PairCorrect"],96],
SameQ[summary86F84["PredictionRelationsCorrect"],96],
SameQ[summary86F84["ScenarioPerfect"],24]
];

resultPayload86F84=<|
"Stage"->"S86F84",
"Name"->"RevealedS84DoubleInterventionRegression",
"CandidateHash"->candidateHashAfter86F84,
"ProtocolHash"->protocolHashAfter86F84,
"Depths"->blindDepths86F84,
"Topologies"->blindTopologies86F84,
"Scenarios"->summary86F84["Scenarios"],
"WorldPairs"->summary86F84["WorldPairs"],
"Worlds"->summary86F84["Worlds"],
"BaselineCorrect"->summary86F84["BaselineCorrect"],
"InterventionContinueCorrect"->summary86F84["InterventionContinueCorrect"],
"InterventionStopCorrect"->summary86F84["InterventionStopCorrect"],
"WorldCorrect"->summary86F84["WorldCorrect"],
"PairCorrect"->summary86F84["PairCorrect"],
"PredictionRelationsCorrect"->summary86F84["PredictionRelationsCorrect"],
"ScenarioPerfect"->summary86F84["ScenarioPerfect"],
"OriginalFrozenModelChanged"->!TrueQ[originalFrozenModelUnchanged86F84],
"FrozenCandidateChanged"->!TrueQ[frozenCandidateUnchanged86F84],
"CoreChanged"->!TrueQ[coreUnchanged86F84],
"CanonicalizerChanged"->!TrueQ[canonicalizerUnchanged86F84],
"InterventionChanged"->!TrueQ[interventionUnchanged86F84],
"TopologyImplementationsChanged"->!TrueQ[topologiesUnchanged86F84],
"DeduplicationMechanismChanged"->!TrueQ[deduplicationMechanismUnchanged86F84],
"TestValidityPassed"->testValidityPassed86F84,
"RegressionPerfect"->regressionPerfect86F84
|>;

regressionResultHash86F84=Hash[
Normal[resultPayload86F84],"SHA256","HexString"
];

cert86F84=Join[
resultPayload86F84,
<|
"CandidateFrozenBeforeS86F84"->True,
"TrainingRun"->False,
"CandidateSearchRun"->False,
"PolicyEditApplied"->False,
"RetuningApplied"->False,
"HistoricalRegressionRerun"->True,
"S83BlindRerun"->False,
"S83BDevelopmentRowsRerun"->False,
"S86F84LabelsUsedForSelection"->False,
"DoubleInterventionNovel"->True,
"AllQueryRolesTestedPerGraph"->True,
"SameQueryBeforeAfterIntervention"->True,
"RevealedRegressionNotBlind"->True,
"PreservesOriginalS84CounterfactualComposition"->regressionPerfect86F84,
"MayClaimGeneralCounterfactualReasoning"->False,
"MayClaimCausalDiscovery"->False,
"TotalTraceSeconds"->summary86F84["TotalTraceSeconds"],
"BlindResultHash"->regressionResultHash86F84,
"Outcome"->Which[
!TrueQ[testValidityPassed86F84],
"INVALID_S86F_S84_REGRESSION",
TrueQ[regressionPerfect86F84],
"S86F_S84_REVEALED_REGRESSION_PASS",
True,
"S86F_S84_VALID_REGRESSION_FAILURE"
],
"SuggestedNextStage"->If[
TrueQ[regressionPerfect86F84],
"S86F_CONTINUE_TO_S85_REVEALED_REGRESSION",
"S86F_STOP_AND_AUDIT_S84_REGRESSION"
]
|>
];

Dataset[{cert86F84}]


In [ ]:
ClearAll[
NodeRole86F85,
EncodePair86F85,
PredictTokens86F85,
SetAnswer86F85,
TopologyTransform86F85,
ExpectedContractions86F85,
BranchContinuePatch86F85,
DoubleRestorePatch86F85,
PrepareWorld86F85,
PrepareScenario86F85,
S86F85TestDefinitionBundle
];

NodeRole86F85[originalNode_,case_List,answer_Integer]:=Module[
{x,m,correct,wrong,dummy,querySources,queryBranch,role},
x=case[[1]];
m=x[[6,answer]];
correct=x[[5,answer]];
wrong=x[[5,1+Mod[answer,4]]];
dummy=m+3;
querySources={m,m+1,m+2};
queryBranch=Union[querySources,{correct,wrong,dummy}];
role=Which[
SameQ[originalNode,m],"QueriedDecision",
MemberQ[querySources,originalNode],"QueriedMediatorSource",
SameQ[originalNode,correct],"QueriedCorrectDestination",
SameQ[originalNode,wrong],"QueriedWrongDestination",
SameQ[originalNode,dummy],"QueriedDummyDestination",
MemberQ[x[[6]],originalNode],"OtherDecision",
MemberQ[x[[5]],originalNode],"OtherAnswerDestination",
True,"OtherReject"
];
<|
"Role"->role,
"QueryBranchRelated"->MemberQ[queryBranch,originalNode]
|>
];

EncodePair86F85[pair_List]:=Module[{encoded},
encoded=First@EncodeRows75[
{<|
"Grammar"->"S86F85BlindObservation",
"Depth"->0,"Answer"->0,"Target"->"Unlabeled",
"StatePairs"->{pair}
|>},
frozenCandidate86E["EncoderParams"],
frozenCandidate86E["K"]
];
First[encoded["Codes"]]
];

PredictTokens86F85[tokens_List]:=If[
AnyTrue[tokens,MemberQ[frozenCandidate86E["Policy"],#]&],
"Continue",
"Stop"
];

SetAnswer86F85[c_List,answer_Integer]:={c[[1]],answer};

TopologyTransform86F85[topology_String,c_List]:=Switch[
topology,
"DiamondIn",DiamondIn72[c],
"DoubleDiamondIn",DoubleDiamondIn79[c],
_,$Failed
];

ExpectedContractions86F85[topology_String,baseCase_List]:=Switch[
topology,
"DiamondIn",DecisionIncomingEdgeCount79B[baseCase],
"DoubleDiamondIn",2 DecisionIncomingEdgeCount79B[baseCase],
_,Missing["UnknownTopology"]
];

BranchContinuePatch86F85[c_List,branch_Integer]:=Module[
{x,e,m,safe,u,dummy,correct,wrong,remove,add},
x=c[[1]];
e=x[[1]];
m=x[[6,branch]];
safe=m+1;
u=m+2;
dummy=m+3;
correct=x[[5,branch]];
wrong=x[[5,1+Mod[branch,4]]];
remove={
DirectedEdge[m,wrong],
DirectedEdge[safe,correct],
DirectedEdge[u,dummy]
};
add={
DirectedEdge[m,correct],
DirectedEdge[safe,dummy],
DirectedEdge[u,wrong]
};
<|
"Remove"->remove,
"Add"->add,
"ValidOnInput"->And[
And@@Map[MemberQ[e,#]&,remove],
And@@Map[!MemberQ[e,#]&,add],
Intersection[remove,add]==={}
]
|>
];

DoubleRestorePatch86F85[c_List,branches_List]:=Module[
{parts,remove,add},
parts=BranchContinuePatch86F85[c,#]&/@branches;
remove=DeleteDuplicates@Flatten[Lookup[parts,"Remove"],1];
add=DeleteDuplicates@Flatten[Lookup[parts,"Add"],1];
<|
"Remove"->remove,
"Add"->add,
"Branches"->branches,
"ComponentPatchesValid"->And@@Lookup[parts,"ValidOnInput"],
"NoCrossBranchConflict"->Intersection[remove,add]==={},
"ExpectedEditCount"->And[Length[remove]===6,Length[add]===6]
|>
];

PrepareWorld86F85[
topology_String,
depth_Integer,
restoredBranches_List,
graphCondition_String,
answer_Integer,
target_String,
baseCase_List
]:=Module[
{
topologyCase,canonicalization,canonicalCase,expectedContractions,
traceSeconds,trace,levels,pack,vertexList,packedNodes,
observations,originalNode,pair,roleInfo,rawTokens,tokens,prediction
},
topologyCase=TopologyTransform86F85[topology,baseCase];
canonicalization=CanonicalizePrivateDiamonds79B[topologyCase];
canonicalCase=canonicalization["Case"];
expectedContractions=ExpectedContractions86F85[topology,baseCase];
{traceSeconds,trace}=AbsoluteTiming[RejectTrace78[canonicalCase]];
levels=SigLevels61[canonicalCase,3];
pack=Pack60[canonicalCase];
vertexList=pack[[12]];
packedNodes=If[
Length[trace["Rejects"]]===0,
{},
DeleteDuplicates[trace["Rejects"][[All,2]]]
];
observations=Map[
Function[packedNode,
originalNode=vertexList[[packedNode]];
pair={Lookup[levels[[3]],packedNode],Lookup[levels[[4]],packedNode]};
roleInfo=NodeRole86F85[originalNode,canonicalCase,answer];
<|
"Role"->roleInfo["Role"],
"QueryBranchRelated"->roleInfo["QueryBranchRelated"],
"Code"->EncodePair86F85[pair]
|>
],
packedNodes
];
rawTokens=({#1["Role"],#1["Code"]}&)/@observations;
tokens=DeleteDuplicates[rawTokens];
prediction=PredictTokens86F85[tokens];
<|
"Topology"->topology,
"Depth"->depth,
"RestoredBranches"->restoredBranches,
"GraphCondition"->graphCondition,
"Answer"->answer,
"Target"->target,
"ReferenceAction"->ReferenceAction82[canonicalCase],
"Prediction"->prediction,
"Correct"->SameQ[prediction,target],
"TopologyGraphHash"->Hash[topologyCase[[1,1]],"SHA256","HexString"],
"CanonicalGraphHash"->Hash[canonicalCase[[1,1]],"SHA256","HexString"],
"CanonicalCaseExactlyBase"->SameQ[canonicalCase,baseCase],
"Contractions"->canonicalization["Contractions"],
"ExpectedContractions"->expectedContractions,
"ContractionCountCorrect"->SameQ[
canonicalization["Contractions"],expectedContractions
],
"ProtectedNodesPreserved"->canonicalization["ProtectedNodesPreserved"],
"StateObservationCount"->Length[observations],
"RawTokenCount"->Length[rawTokens],
"TokenCount"->Length[tokens],
"DuplicateTokensRemoved"->Length[rawTokens]-Length[tokens],
"PolicyHitTokens"->Intersection[tokens,frozenCandidate86E["Policy"]],
"TerminatedNaturally"->trace["TerminatedNaturally"],
"HitSafetyCap"->trace["HitSafetyCap"],
"Rounds"->trace["Rounds"],
"TraceSeconds"->traceSeconds
|>
];

PrepareScenario86F85[
topology_String,depth_Integer,restoredBranches_List
]:=Module[
{
seedCase,patch,hybridSeed,baseWorlds,hybridWorlds,
worldPairs,baseGraphHashes,hybridGraphHashes
},
seedCase=Case59[depth,1,"Stop"];
patch=DoubleRestorePatch86F85[seedCase,restoredBranches];
hybridSeed=ApplyEdgePatch81[seedCase,patch];
If[SameQ[hybridSeed,$Failed],Return[$Failed]];
baseWorlds=Table[
PrepareWorld86F85[
topology,depth,restoredBranches,"Baseline",answer,"Stop",
SetAnswer86F85[seedCase,answer]
],
{answer,Range[4]}
];
hybridWorlds=Table[
PrepareWorld86F85[
topology,depth,restoredBranches,"InverseDoubleRestoration",answer,
If[MemberQ[restoredBranches,answer],"Continue","Stop"],
SetAnswer86F85[hybridSeed,answer]
],
{answer,Range[4]}
];
worldPairs=MapThread[
Function[{base,hybrid},
<|
"Answer"->base["Answer"],
"RestoredQuery"->MemberQ[restoredBranches,base["Answer"]],
"SameQuery"->SameQ[base["Answer"],hybrid["Answer"]],
"ReferenceRelationCorrect"->If[
MemberQ[restoredBranches,base["Answer"]],
And[
SameQ[base["ReferenceAction"],"Stop"],
SameQ[hybrid["ReferenceAction"],"Continue"]
],
And[
SameQ[base["ReferenceAction"],"Stop"],
SameQ[hybrid["ReferenceAction"],"Stop"]
]
],
"PredictionRelationCorrect"->If[
MemberQ[restoredBranches,base["Answer"]],
And[
SameQ[base["Prediction"],"Stop"],
SameQ[hybrid["Prediction"],"Continue"]
],
And[
SameQ[base["Prediction"],"Stop"],
SameQ[hybrid["Prediction"],"Stop"]
]
],
"PairCorrect"->And[TrueQ[base["Correct"]],TrueQ[hybrid["Correct"]]],
"BaselineWorld"->base,
"InterventionWorld"->hybrid
|>
],
{baseWorlds,hybridWorlds}
];
baseGraphHashes=Lookup[baseWorlds,"TopologyGraphHash"];
hybridGraphHashes=Lookup[hybridWorlds,"TopologyGraphHash"];
<|
"Topology"->topology,
"Depth"->depth,
"RestoredBranches"->restoredBranches,
"RestoreComponentValidity"->patch["ComponentPatchesValid"],
"RestoreNoConflict"->patch["NoCrossBranchConflict"],
"RestoreEditCountCorrect"->patch["ExpectedEditCount"],
"BaselineSameGraphAcrossQueries"->SameQ@@baseGraphHashes,
"InterventionSameGraphAcrossQueries"->SameQ@@hybridGraphHashes,
"RestorationChangesGraph"->UnsameQ[First[baseGraphHashes],First[hybridGraphHashes]],
"ReferenceRelationsCorrect"->And@@Lookup[worldPairs,"ReferenceRelationCorrect"],
"PredictionRelationsCorrect"->And@@Lookup[worldPairs,"PredictionRelationCorrect"],
"AllEightWorldsCorrect"->And@@Join[
Lookup[baseWorlds,"Correct"],Lookup[hybridWorlds,"Correct"]
],
"WorldPairs"->worldPairs,
"BaselineWorlds"->baseWorlds,
"InterventionWorlds"->hybridWorlds
|>
];

S86F85TestDefinitionBundle[]:={
DownValues[NodeRole86F85],DownValues[EncodePair86F85],DownValues[PredictTokens86F85],
DownValues[SetAnswer86F85],DownValues[TopologyTransform86F85],
DownValues[ExpectedContractions86F85],DownValues[BranchContinuePatch86F85],
DownValues[DoubleRestorePatch86F85],DownValues[PrepareWorld86F85],
DownValues[PrepareScenario86F85]
};

blindDepths86F85={31,59};
blindTopologies86F85={"DiamondIn","DoubleDiamondIn"};
blindRestoredBranchPairs86F85=Subsets[Range[4],{2}];

protocol86F85=<|
"Stage"->"S86F85",
"Name"->"RevealedS85InverseInterventionRegression",
"Candidate"->"S86E-K33ExactRole",
"CandidateHash"->candidateHashLoaded86F85,
"Depths"->blindDepths86F85,
"Topologies"->blindTopologies86F85,
"RestoredBranchPairs"->blindRestoredBranchPairs86F85,
"ExpectedScenarios"->24,
"ExpectedWorldPairs"->96,
"ExpectedWorlds"->192,
"Intervention"->"TwoSimultaneousInverseBranchRestorations",
"QueryGrid"->"AllFourQueriesBeforeAndAfterInverseIntervention",
"ExpectedRestoredQueryPairs"->48,
"ExpectedUnrestoredQueryPairs"->48,
"TokenDeduplication"->"DeleteDuplicatesAfterExactRoleCodePairing",
"CandidateFrozenBeforeProtocol"->True,
"CandidateSearchRun"->False,
"TrainingRun"->False,
"HistoricalRegressionRerun"->True,
"S83BlindRerun"->False,
"S84BlindRerun"->False,
"S83BDevelopmentRowsRerun"->False,
"S86F85LabelsUsedForSelection"->False,
"NoCaseEvaluatedBeforeProtocolHash"->True
|>;

protocolHash86F85=Hash[Normal[protocol86F85],"SHA256","HexString"];
modelHashBefore86F85=Hash[Normal[frozen75D],"SHA256","HexString"];
candidateHashBefore86F85=Hash[Normal[frozenCandidate86E],"SHA256","HexString"];
coreHashBefore86F85=Hash[CoreDefinitionBundle86F85[],"SHA256","HexString"];
canonicalizerHashBefore86F85=canonicalizerImplementationHash79B;
interventionHashBefore86F85=interventionImplementationHash82;
topologyHashBefore86F85=Hash[
{DownValues[DiamondIn72],DownValues[DoubleDiamondIn79]},
"SHA256","HexString"
];
testDefinitionHashBefore86F85=Hash[
S86F85TestDefinitionBundle[],"SHA256","HexString"
];

Dataset[{Join[protocol86F85,<|"ProtocolHash"->protocolHash86F85|>]}]


In [ ]:
blindScenarios86F85=Flatten[
Table[
PrepareScenario86F85[topology,depth,restoredBranches],
{topology,blindTopologies86F85},
{depth,blindDepths86F85},
{restoredBranches,blindRestoredBranchPairs86F85}
],
2
];

blindWorldPairs86F85=Flatten[Lookup[blindScenarios86F85,"WorldPairs"],1];
baselineWorlds86F85=Flatten[Lookup[blindScenarios86F85,"BaselineWorlds"],1];
interventionWorlds86F85=Flatten[
Lookup[blindScenarios86F85,"InterventionWorlds"],1
];
blindWorlds86F85=Join[baselineWorlds86F85,interventionWorlds86F85];

summary86F85=<|
"Scenarios"->Length[blindScenarios86F85],
"WorldPairs"->Length[blindWorldPairs86F85],
"Worlds"->Length[blindWorlds86F85],
"RestoredQueryPairs"->Count[
blindWorldPairs86F85,p_/;TrueQ[p["RestoredQuery"]]
],
"UnrestoredQueryPairs"->Count[
blindWorldPairs86F85,p_/;!TrueQ[p["RestoredQuery"]]
],
"RestoreComponentValidity"->Count[
blindScenarios86F85,s_/;TrueQ[s["RestoreComponentValidity"]]
],
"RestoreNoConflict"->Count[
blindScenarios86F85,s_/;TrueQ[s["RestoreNoConflict"]]
],
"RestoreEditCountCorrect"->Count[
blindScenarios86F85,s_/;TrueQ[s["RestoreEditCountCorrect"]]
],
"BaselineSameGraphAcrossQueries"->Count[
blindScenarios86F85,s_/;TrueQ[s["BaselineSameGraphAcrossQueries"]]
],
"InterventionSameGraphAcrossQueries"->Count[
blindScenarios86F85,s_/;TrueQ[s["InterventionSameGraphAcrossQueries"]]
],
"RestorationChangesGraph"->Count[
blindScenarios86F85,s_/;TrueQ[s["RestorationChangesGraph"]]
],
"ReferenceRelationsCorrect"->Count[
blindWorldPairs86F85,p_/;TrueQ[p["ReferenceRelationCorrect"]]
],
"PredictionRelationsCorrect"->Count[
blindWorldPairs86F85,p_/;TrueQ[p["PredictionRelationCorrect"]]
],
"PairCorrect"->Count[
blindWorldPairs86F85,p_/;TrueQ[p["PairCorrect"]]
],
"ScenarioPerfect"->Count[
blindScenarios86F85,s_/;TrueQ[s["AllEightWorldsCorrect"]]
],
"BaselineCorrect"->Count[baselineWorlds86F85,w_/;TrueQ[w["Correct"]]],
"InterventionContinueCorrect"->Count[
interventionWorlds86F85,
w_/;SameQ[w["Target"],"Continue"]&&TrueQ[w["Correct"]]
],
"InterventionStopCorrect"->Count[
interventionWorlds86F85,
w_/;SameQ[w["Target"],"Stop"]&&TrueQ[w["Correct"]]
],
"WorldCorrect"->Count[blindWorlds86F85,w_/;TrueQ[w["Correct"]]],
"CanonicalCaseExactlyBase"->Count[
blindWorlds86F85,w_/;TrueQ[w["CanonicalCaseExactlyBase"]]
],
"ContractionCountCorrect"->Count[
blindWorlds86F85,w_/;TrueQ[w["ContractionCountCorrect"]]
],
"ProtectedNodesPreserved"->Count[
blindWorlds86F85,w_/;TrueQ[w["ProtectedNodesPreserved"]]
],
"ReferenceActionsCorrect"->Count[
blindWorlds86F85,w_/;SameQ[w["ReferenceAction"],w["Target"]]
],
"NonEmptyTokens"->Count[blindWorlds86F85,w_/;w["TokenCount"]>0],
"TerminatedNaturally"->Count[
blindWorlds86F85,w_/;TrueQ[w["TerminatedNaturally"]]
],
"HitSafetyCap"->Count[
blindWorlds86F85,w_/;TrueQ[w["HitSafetyCap"]]
],
"TotalTraceSeconds"->Total@Lookup[blindWorlds86F85,"TraceSeconds"]
|>;

byTopology86F85=Map[
Function[topology,
Module[{scenarios,pairs,base,intervention,worlds},
scenarios=Select[blindScenarios86F85,SameQ[#["Topology"],topology]&];
pairs=Flatten[Lookup[scenarios,"WorldPairs"],1];
base=Flatten[Lookup[scenarios,"BaselineWorlds"],1];
intervention=Flatten[Lookup[scenarios,"InterventionWorlds"],1];
worlds=Join[base,intervention];
<|
"Topology"->topology,
"Scenarios"->Length[scenarios],
"Worlds"->Length[worlds],
"BaselineCorrect"->Count[base,w_/;TrueQ[w["Correct"]]],
"InterventionContinueCorrect"->Count[
intervention,w_/;SameQ[w["Target"],"Continue"]&&TrueQ[w["Correct"]]
],
"InterventionStopCorrect"->Count[
intervention,w_/;SameQ[w["Target"],"Stop"]&&TrueQ[w["Correct"]]
],
"PairCorrect"->Count[pairs,p_/;TrueQ[p["PairCorrect"]]],
"ScenarioPerfect"->Count[
scenarios,s_/;TrueQ[s["AllEightWorldsCorrect"]]
],
"CanonicalExact"->Count[
worlds,w_/;TrueQ[w["CanonicalCaseExactlyBase"]]
],
"TerminatedNaturally"->Count[
worlds,w_/;TrueQ[w["TerminatedNaturally"]]
],
"TraceSeconds"->Total@Lookup[worlds,"TraceSeconds"]
|>
]
],
blindTopologies86F85
];

Column[{
Dataset[Map[
KeyTake[#,{"Topology","Depth","RestoredBranches",
"RestoreComponentValidity","RestoreNoConflict","RestoreEditCountCorrect",
"BaselineSameGraphAcrossQueries","InterventionSameGraphAcrossQueries",
"ReferenceRelationsCorrect","PredictionRelationsCorrect",
"AllEightWorldsCorrect"}]&,
blindScenarios86F85
]],
Dataset[byTopology86F85],
Dataset[{summary86F85}]
}]


In [ ]:
modelHashAfter86F85=Hash[Normal[frozen75D],"SHA256","HexString"];
candidateHashAfter86F85=Hash[
Normal[frozenCandidate86E],"SHA256","HexString"
];
coreHashAfter86F85=Hash[CoreDefinitionBundle86F85[],"SHA256","HexString"];
canonicalizerHashAfter86F85=Hash[
{
DownValues[FindPrivateDiamond79B],
DownValues[CanonicalizePrivateDiamonds79B],
DownValues[CanonicalCase79B]
},
"SHA256","HexString"
];
interventionHashAfter86F85=Hash[
{
DownValues[LocalMediatorSources82],
DownValues[FullSemanticPatch82],
DownValues[LocalMediatorPatch82],
DownValues[ReferenceAction82]
},
"SHA256","HexString"
];
topologyHashAfter86F85=Hash[
{DownValues[DiamondIn72],DownValues[DoubleDiamondIn79]},
"SHA256","HexString"
];
testDefinitionHashAfter86F85=Hash[
S86F85TestDefinitionBundle[],"SHA256","HexString"
];
protocolHashAfter86F85=Hash[Normal[protocol86F85],"SHA256","HexString"];

originalFrozenModelUnchanged86F85=And[
SameQ[modelHashBefore86F85,modelHashAfter86F85],
SameQ[modelHashAfter86F85,expectedFrozenModelHash79A]
];
frozenCandidateUnchanged86F85=And[
SameQ[candidateHashBefore86F85,candidateHashAfter86F85],
SameQ[candidateHashAfter86F85,expectedCandidateHash86F85]
];
coreUnchanged86F85=SameQ[coreHashBefore86F85,coreHashAfter86F85];
canonicalizerUnchanged86F85=And[
SameQ[canonicalizerHashBefore86F85,canonicalizerHashAfter86F85],
SameQ[canonicalizerHashAfter86F85,expectedCanonicalizerHash86F85]
];
interventionUnchanged86F85=And[
SameQ[interventionHashBefore86F85,interventionHashAfter86F85],
SameQ[interventionHashAfter86F85,expectedInterventionHash86F85]
];
topologiesUnchanged86F85=SameQ[topologyHashBefore86F85,topologyHashAfter86F85];
testDefinitionUnchanged86F85=SameQ[
testDefinitionHashBefore86F85,testDefinitionHashAfter86F85
];
protocolUnchanged86F85=SameQ[protocolHash86F85,protocolHashAfter86F85];
deduplicationMechanismUnchanged86F85=And[
TrueQ[coreUnchanged86F85],
TrueQ[testDefinitionUnchanged86F85],
SameQ[protocol86F85["TokenDeduplication"],
"DeleteDuplicatesAfterExactRoleCodePairing"]
];

testValidityPassed86F85=And[
TrueQ[preflightPassed86F85],
TrueQ[originalFrozenModelUnchanged86F85],
TrueQ[frozenCandidateUnchanged86F85],
TrueQ[coreUnchanged86F85],
TrueQ[canonicalizerUnchanged86F85],
TrueQ[interventionUnchanged86F85],
TrueQ[topologiesUnchanged86F85],
TrueQ[testDefinitionUnchanged86F85],
TrueQ[protocolUnchanged86F85],
TrueQ[deduplicationMechanismUnchanged86F85],
SameQ[summary86F85["Scenarios"],24],
SameQ[summary86F85["WorldPairs"],96],
SameQ[summary86F85["Worlds"],192],
SameQ[summary86F85["RestoredQueryPairs"],48],
SameQ[summary86F85["UnrestoredQueryPairs"],48],
SameQ[summary86F85["RestoreComponentValidity"],24],
SameQ[summary86F85["RestoreNoConflict"],24],
SameQ[summary86F85["RestoreEditCountCorrect"],24],
SameQ[summary86F85["BaselineSameGraphAcrossQueries"],24],
SameQ[summary86F85["InterventionSameGraphAcrossQueries"],24],
SameQ[summary86F85["RestorationChangesGraph"],24],
SameQ[summary86F85["ReferenceRelationsCorrect"],96],
SameQ[summary86F85["CanonicalCaseExactlyBase"],192],
SameQ[summary86F85["ContractionCountCorrect"],192],
SameQ[summary86F85["ProtectedNodesPreserved"],192],
SameQ[summary86F85["ReferenceActionsCorrect"],192],
SameQ[summary86F85["NonEmptyTokens"],192],
SameQ[summary86F85["TerminatedNaturally"],192],
SameQ[summary86F85["HitSafetyCap"],0]
];

regressionPerfect86F85=And[
TrueQ[testValidityPassed86F85],
SameQ[summary86F85["BaselineCorrect"],96],
SameQ[summary86F85["InterventionContinueCorrect"],48],
SameQ[summary86F85["InterventionStopCorrect"],48],
SameQ[summary86F85["WorldCorrect"],192],
SameQ[summary86F85["PairCorrect"],96],
SameQ[summary86F85["PredictionRelationsCorrect"],96],
SameQ[summary86F85["ScenarioPerfect"],24]
];

resultPayload86F85=<|
"Stage"->"S86F85",
"Name"->"RevealedS85InverseInterventionRegression",
"CandidateHash"->candidateHashAfter86F85,
"ProtocolHash"->protocolHashAfter86F85,
"Depths"->blindDepths86F85,
"Topologies"->blindTopologies86F85,
"Scenarios"->summary86F85["Scenarios"],
"WorldPairs"->summary86F85["WorldPairs"],
"Worlds"->summary86F85["Worlds"],
"BaselineCorrect"->summary86F85["BaselineCorrect"],
"InterventionContinueCorrect"->summary86F85["InterventionContinueCorrect"],
"InterventionStopCorrect"->summary86F85["InterventionStopCorrect"],
"WorldCorrect"->summary86F85["WorldCorrect"],
"PairCorrect"->summary86F85["PairCorrect"],
"PredictionRelationsCorrect"->summary86F85["PredictionRelationsCorrect"],
"ScenarioPerfect"->summary86F85["ScenarioPerfect"],
"OriginalFrozenModelChanged"->!TrueQ[originalFrozenModelUnchanged86F85],
"FrozenCandidateChanged"->!TrueQ[frozenCandidateUnchanged86F85],
"CoreChanged"->!TrueQ[coreUnchanged86F85],
"CanonicalizerChanged"->!TrueQ[canonicalizerUnchanged86F85],
"InterventionChanged"->!TrueQ[interventionUnchanged86F85],
"TopologyImplementationsChanged"->!TrueQ[topologiesUnchanged86F85],
"DeduplicationMechanismChanged"->!TrueQ[deduplicationMechanismUnchanged86F85],
"TestValidityPassed"->testValidityPassed86F85,
"RegressionPerfect"->regressionPerfect86F85
|>;

regressionResultHash86F85=Hash[
Normal[resultPayload86F85],"SHA256","HexString"
];

cert86F85=Join[
resultPayload86F85,
<|
"CandidateFrozenBeforeS86F85"->True,
"TrainingRun"->False,
"CandidateSearchRun"->False,
"PolicyEditApplied"->False,
"RetuningApplied"->False,
"HistoricalRegressionRerun"->True,
"S83BlindRerun"->False,
"S84BlindRerun"->False,
"S83BDevelopmentRowsRerun"->False,
"S86F85LabelsUsedForSelection"->False,
"InverseInterventionDirectionNovel"->True,
"AllQueryRolesTestedPerGraph"->True,
"SameQueryBeforeAfterIntervention"->True,
"RevealedRegressionNotBlind"->True,
"PreservesOriginalS85CounterfactualComposition"->regressionPerfect86F85,
"MayClaimGeneralCounterfactualReasoning"->False,
"MayClaimCausalDiscovery"->False,
"TotalTraceSeconds"->summary86F85["TotalTraceSeconds"],
"BlindResultHash"->regressionResultHash86F85,
"Outcome"->Which[
!TrueQ[testValidityPassed86F85],
"INVALID_S86F_S85_REGRESSION",
TrueQ[regressionPerfect86F85],
"S86F_S85_REVEALED_REGRESSION_PASS",
True,
"S86F_S85_VALID_REGRESSION_FAILURE"
],
"SuggestedNextStage"->If[
TrueQ[regressionPerfect86F85],
"S86F85_INDEPENDENT_INTERVENTION_OPERATOR_BLIND_TEST",
"S86F_STOP_AND_AUDIT_S85_REGRESSION"
]
|>
];

Dataset[{cert86F85}]


In [ ]:
modelHashAfter86F=Hash[Normal[frozen75D],"SHA256","HexString"];
baseK19CandidateHashAfter86F=Hash[
Normal[frozenCandidate83B],"SHA256","HexString"
];
k33CandidateHashAfter86F=Hash[
Normal[frozenCandidate86E],"SHA256","HexString"
];
coreHashAfter86F=Hash[CoreDefinitionBundle86[],"SHA256","HexString"];
canonicalizerHashAfter86F=canonicalizerImplementationHash79B;
interventionHashAfter86F=interventionImplementationHash82;
k33CandidateFileHashAfter86F=FileHash[k33CandidatePath86F,"SHA256"];
oldK19CandidateFileHashAfter86F=FileHash[oldK19CandidatePath86F,"SHA256"];

originalFrozenModelUnchanged86F=SameQ[
modelHashBefore86F,modelHashAfter86F
];
baseK19CandidateUnchanged86F=And[
SameQ[baseK19CandidateHashBefore86F,baseK19CandidateHashAfter86F],
SameQ[baseK19CandidateHashAfter86F,expectedBaseK19CandidateHash86F],
SameQ[oldK19CandidateFileHashBefore86F,oldK19CandidateFileHashAfter86F]
];
k33CandidateUnchanged86F=And[
SameQ[k33CandidateHashBefore86F,k33CandidateHashAfter86F],
SameQ[k33CandidateHashAfter86F,expectedK33CandidateHash86F],
SameQ[k33CandidateFileHashBefore86F,k33CandidateFileHashAfter86F]
];
coreUnchanged86F=SameQ[coreHashBefore86F,coreHashAfter86F];
canonicalizerUnchanged86F=SameQ[
canonicalizerHashBefore86F,canonicalizerHashAfter86F
];
interventionUnchanged86F=SameQ[
interventionHashBefore86F,interventionHashAfter86F
];

regressionValidityPassed86F=And[
TrueQ[preflightPassed86F],
TrueQ[testValidityPassed86F84],
TrueQ[testValidityPassed86F85],
TrueQ[originalFrozenModelUnchanged86F],
TrueQ[baseK19CandidateUnchanged86F],
TrueQ[k33CandidateUnchanged86F],
TrueQ[coreUnchanged86F],
TrueQ[canonicalizerUnchanged86F],
TrueQ[interventionUnchanged86F],
SameQ[summary86F84["Scenarios"],24],
SameQ[summary86F84["WorldPairs"],96],
SameQ[summary86F84["Worlds"],192],
SameQ[summary86F85["Scenarios"],24],
SameQ[summary86F85["WorldPairs"],96],
SameQ[summary86F85["Worlds"],192]
];

regressionPerfect86F=And[
TrueQ[regressionValidityPassed86F],
TrueQ[regressionPerfect86F84],
TrueQ[regressionPerfect86F85],
SameQ[summary86F84["WorldCorrect"],192],
SameQ[summary86F85["WorldCorrect"],192],
SameQ[summary86F84["PairCorrect"],96],
SameQ[summary86F85["PairCorrect"],96],
SameQ[summary86F84["ScenarioPerfect"],24],
SameQ[summary86F85["ScenarioPerfect"],24]
];

resultPayload86F=<|
"Stage"->"S86F",
"Name"->"RevealedS84S85Regression",
"AuditType"->"RevealedRegressionNotBlind",
"CandidateHash"->k33CandidateHashAfter86F,
"K"->frozenCandidate86E["K"],
"PolicyLength"->frozenCandidate86E["PolicyLength"],
"S84Scenarios"->summary86F84["Scenarios"],
"S84WorldPairs"->summary86F84["WorldPairs"],
"S84WorldCorrect"->summary86F84["WorldCorrect"],
"S84PairCorrect"->summary86F84["PairCorrect"],
"S84ScenarioPerfect"->summary86F84["ScenarioPerfect"],
"S84RegressionPassed"->regressionPerfect86F84,
"S85Scenarios"->summary86F85["Scenarios"],
"S85WorldPairs"->summary86F85["WorldPairs"],
"S85WorldCorrect"->summary86F85["WorldCorrect"],
"S85PairCorrect"->summary86F85["PairCorrect"],
"S85ScenarioPerfect"->summary86F85["ScenarioPerfect"],
"S85RegressionPassed"->regressionPerfect86F85,
"CombinedScenarios"->summary86F84["Scenarios"]+summary86F85["Scenarios"],
"CombinedWorldPairs"->summary86F84["WorldPairs"]+summary86F85["WorldPairs"],
"CombinedWorlds"->summary86F84["Worlds"]+summary86F85["Worlds"],
"CombinedWorldCorrect"->
summary86F84["WorldCorrect"]+summary86F85["WorldCorrect"],
"OriginalFrozenModelChanged"->!TrueQ[originalFrozenModelUnchanged86F],
"BaseK19CandidateChanged"->!TrueQ[baseK19CandidateUnchanged86F],
"K33CandidateChanged"->!TrueQ[k33CandidateUnchanged86F],
"CoreChanged"->!TrueQ[coreUnchanged86F],
"CanonicalizerChanged"->!TrueQ[canonicalizerUnchanged86F],
"InterventionChanged"->!TrueQ[interventionUnchanged86F],
"S84TopologyImplementationsChanged"->cert86F84["TopologyImplementationsChanged"],
"S85TopologyImplementationsChanged"->cert86F85["TopologyImplementationsChanged"],
"DeduplicationMechanismChanged"->Or[
TrueQ[cert86F84["DeduplicationMechanismChanged"]],
TrueQ[cert86F85["DeduplicationMechanismChanged"]]
],
"TrainingRun"->False,
"CandidateSelectionRun"->False,
"PolicyEditApplied"->False,
"RetuningApplied"->False,
"S84S85LabelsUsedForSelection"->False,
"S87DataUsed"->False,
"RegressionValidityPassed"->regressionValidityPassed86F,
"RegressionPerfect"->regressionPerfect86F,
"TotalTraceSeconds"->
summary86F84["TotalTraceSeconds"]+summary86F85["TotalTraceSeconds"]
|>;

regressionResultHash86F=Hash[
Normal[resultPayload86F],"SHA256","HexString"
];

cert86F=Join[
resultPayload86F,
<|
"RegressionResultHash"->regressionResultHash86F,
"Outcome"->Which[
!TrueQ[regressionValidityPassed86F],
"INVALID_S86F_REGRESSION",
TrueQ[regressionPerfect86F],
"S86F_REVEALED_S84_S85_REGRESSION_PASS",
True,
"VALID_S86F_REGRESSION_FAILURE"
],
"SuggestedNextStage"->If[
TrueQ[regressionPerfect86F],
"S87_NEW_BLIND_TEST_WITH_FROZEN_K33",
"S86G_FAILURE_AUDIT_WITHOUT_RETUNING"
]
|>
];

Dataset[{cert86F}]
